In [1]:
!pip install --upgrade huggingface_hub


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 553.3/553.3 kB 10.6 MB/s eta 0:00:00
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.4.0
    Uninstalling huggingface_hub-1.4.0:
      Successfully uninstalled huggingface_hub-1.4.0


In [2]:
from google.colab import userdata

In [3]:
READ_TOKEN = userdata.get("HF_TOKEN_READ")

In [5]:
WRITE_TOKEN = userdata.get("HF_TOKEN_WRITE")

In [7]:
import os
from huggingface_hub import HfApi
from google.colab import userdata

In [53]:
api = HfApi(token = WRITE_TOKEN)

In [54]:
api.create_repo(
    repo_id = "SANGAMESWAR/sample",
    repo_type = "model",
    private = False
)

RepoUrl('https://huggingface.co/SANGAMESWAR/sample', endpoint='https://huggingface.co', repo_type='model', repo_id='SANGAMESWAR/sample')

In [8]:
!pip install -U datasets fsspec

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.2/515.2 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.0/201.0 kB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 20.2 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.0
    Uninstalling fsspec-2025.3.0:
      Successfully uninstalled fsspec-2025.3.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2025.10.0 which is incompatible.


## DATASET

In [9]:
from datasets import load_dataset

In [10]:
dataset = load_dataset("stanfordnlp/imdb")

README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [11]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})


In [12]:
print(dataset["train"][0])

{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far be

In [14]:
print(dataset["train"].features)
print(dataset["train"].column_names)
print(dataset["train"].shape)
print(dataset["train"].num_rows)
print(dataset["train"].num_columns)

{'text': Value('string'), 'label': ClassLabel(names=['neg', 'pos'])}
['text', 'label']
(25000, 2)
25000
2


In [20]:
shuffled = dataset["train"].shuffle()

In [21]:
shuffled

Dataset({
    features: ['text', 'label'],
    num_rows: 25000
})

In [22]:
shuffled = dataset["train"].shuffle(seed=42)

In [23]:
shuffled

Dataset({
    features: ['text', 'label'],
    num_rows: 25000
})

In [18]:
shuffled.select(range(100))

Dataset({
    features: ['text', 'label'],
    num_rows: 100
})

In [25]:
subset_train = dataset["train"].shuffle(seed = 42).select(range(5000))

In [26]:
subset_train

Dataset({
    features: ['text', 'label'],
    num_rows: 5000
})

## PREPROCESSING

In [27]:
dataset["train"]

Dataset({
    features: ['text', 'label'],
    num_rows: 25000
})

In [28]:
short_reviews = dataset["train"].filter(lambda x: len(x["text"])<100)

Filter:   0%|          | 0/25000 [00:00<?, ? examples/s]

In [29]:
short_reviews

Dataset({
    features: ['text', 'label'],
    num_rows: 9
})

In [31]:
short_positive = dataset["train"].filter(lambda x: x["label"]==1 and  len(x["text"])<200)

Filter:   0%|          | 0/25000 [00:00<?, ? examples/s]

In [32]:
short_positive

Dataset({
    features: ['text', 'label'],
    num_rows: 60
})

In [33]:
def add_word(example):
  example["word_count"] = len(example["text"].split())
  return example


In [34]:
dataset = dataset.map(add_word)

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [35]:
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'word_count'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label', 'word_count'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label', 'word_count'],
        num_rows: 50000
    })
})

In [36]:
print(dataset["train"][0]["word_count"])

288


## LOAD LARGE DATASET BY STREAMING

In [38]:
big_data = load_dataset("openwebtext",streaming = True)

README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/80 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/80 [00:00<?, ?it/s]

In [39]:
big_data

IterableDatasetDict({
    train: IterableDataset({
        features: ['text'],
        num_shards: 80
    })
})

In [40]:
for item in big_data["train"].take(5):
  print(item)

{'text': 'Port-au-Prince, Haiti (CNN) -- Earthquake victims, writhing in pain and grasping at life, watched doctors and nurses walk away from a field hospital Friday night after a Belgian medical team evacuated the area, saying it was concerned about security.\n\nThe decision left CNN Chief Medical Correspondent Sanjay Gupta as the only doctor at the hospital to get the patients through the night.\n\nCNN initially reported, based on conversations with some of the doctors, that the United Nations ordered the Belgian First Aid and Support Team to evacuate. However, Belgian Chief Coordinator Geert Gijs, a doctor who was at the hospital with 60 Belgian medical personnel, said it was his decision to pull the team out for the night. Gijs said he requested U.N. security personnel to staff the hospital overnight, but was told that peacekeepers would only be able to evacuate the team.\n\nHe said it was a "tough decision" but that he accepted the U.N. offer to evacuate after a Canadian medical t

## CUSTOM DATASET --> PUSH TO HF

In [41]:
from datasets import Dataset
import pandas as pd

data = {
    "text" : [
        "I love this product! It works great!",
        "It was a terrible experience, and i need a refund",
        "Fast and efficient delivery of ball, Wow!"
    ],
    "label" : [1,0,1]
}

In [43]:
data

{'text': ['I love this product! It works great!',
  'It was a terrible experience, and i need a refund',
  'Fast and efficient delivery of ball, Wow!'],
 'label': [1, 0, 1]}

In [42]:
df = pd.DataFrame(data)

In [44]:
df

,text,label
0,I love this product! It works great!,1
1,"It was a terrible experience, and i need a refund",0
2,"Fast and efficient delivery of ball, Wow!",1


In [45]:
dataset = Dataset.from_pandas(df)

In [46]:
dataset

Dataset({
    features: ['text', 'label'],
    num_rows: 3
})

In [47]:
dataset = dataset.class_encode_column("label")

Stringifying the column:   0%|          | 0/3 [00:00<?, ? examples/s]

Casting to class labels:   0%|          | 0/3 [00:00<?, ? examples/s]

In [48]:
dataset

Dataset({
    features: ['text', 'label'],
    num_rows: 3
})

In [49]:
from huggingface_hub import notebook_login
notebook_login()

## PUSH

In [ ]:
dataset.push_to_hub("test_dataset")

## TOKENIZATION

In [55]:
!pip install transformers

In [56]:
from transformers import AutoTokenizer, AutoModel
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

In [57]:
def tokenize(example):
  return tokenizer(example, truncation=True, padding="max_length")

In [58]:
dataset["text"][0]

'I love this product! It works great!'

In [59]:
tokenize(dataset["text"][0])

{'input_ids': [101, 146, 1567, 1142, 3317, 106, 1135, 1759, 1632, 106, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 

In [66]:
tokens = tokenizer.tokenize("Heloo, how are you",return_tensors="pt")

In [67]:
tokens

['He', '##lo', '##o', ',', 'how', 'are', 'you']

## How to Tokenize large dataset faster

In [ ]:
import time
text = ["This is a sample sentence for tokenization"] * 100000

In [ ]:
## Fast
start = time.time()
tok_fast = AutoTokenizer.from_pretrained("bert-base-cased", use_fast=True)
tok_fast(text, padding = True, truncation = True)
print("Fast Time:", time.time() - start)

In [70]:
## Slow
start = time.time()
tok_slow = AutoTokenizer.from_pretrained("bert-base-cased", use_fast=False)
tok_slow(text, padding = True, truncation = True)
print("Slow Time:", time.time() - start)

Slow Time: 0.4576282501220703


## TRAIN OWN TOKENIZER

In [ ]:
from tokenizer import Tokenizer, models, trainers, pre_tokenizers